# Solving the CVRP using a Multimodal Genetic Algorithm (GA)

In this notebook, we will explore and evaluate different multimodal Genetic Algorithm (GA) configurations to solve the Capacitated Vehicle Routing Problem (CVRP).

## Introduction & Environment Setup

The Capacitated Vehicle Routing Problem (CVRP) is a classic logistics challenge where a fleet of vehicles with limited capacity must service a set of geographically dispersed customers. In real-world scenarios, finding a single 'optimal' route is often insufficient due to unforeseen disruptions, such as traffic or road closures. Therefore, our primary goal is to discover multiple, highly diverse routing alternatives of high quality for each problem instance (a multimodal approach).

In this initial section, we set up our experimental framework. We will load our dataset of CVRP instances and initialize our evolutionary engine and orchestrator classes designed to automate the execution, evaluation, and data logging of the genetic algorithms we will test.

## Why use a Genetic Algorithm?

Genetic Algorithms are powerful, bio-inspired metaheuristics based on the principles of Darwinian evolution and natural selection. Rather than constructing a single solution step-by-step probabilistically, a GA maintains a **population of diverse candidate routes**. 

Through successive generations, these candidate solutions evolve by exchanging genetic material (**crossover**) to inherit good topological traits from parent routes, and by introducing random variations (**mutation**) to explore new areas of the map. This population-based approach makes GAs exceptionally robust global searchers. It allows the algorithm to simultaneously explore multiple regions of the solution space, making it a naturally strong candidate for multimodal optimization where maintaining diverse routing strategies is the ultimate goal.

## Which algorithms are we going to evaluate?

To ensure a rigorous comparison with the swarm intelligence models, we have implemented a highly modular GA engine. We will evaluate our evolutionary approach across two key strategic dimensions to observe their impact on routing efficiency and diversity:

1. **Refinement & exploitation Strategies:**
    * **Pure Genetic Algorithm (Baseline):** Relies solely on classic evolutionary operators (OX1 Crossover and Swap Mutation) to explore the solution space. It acts as our control to measure raw evolutionary search power without local assistance.
    * **Memetic Algorithm (GA + Local Search):** A hybrid approach where a fast local search operator (such as 2-opt) is applied to the offspring. This acts as a localized learning step, allowing candidate routes to intelligently 'untangle' crossed paths before their fitness is evaluated, vastly accelerating convergence.

2. **Multimodal Diversity Techniques:**
    * **Standard Generational (Unimodal):** The algorithm prioritizes absolute fitness during replacement, naturally converging the entire population towards a single 'Global Best' route (risking premature convergence).
    * **Deterministic Crowding (Multimodal):** The algorithm actively protects diversity during the replacement phase using a Jaccard Distance threshold. Offspring only compete against structurally similar individuals. This forces the population to maintain multiple isolated "niches", yielding our required 3 distinct routing alternatives.

In [ ]:
# Libraries to use
import os # OS library
import time

# Core CVRP components shared across all metaheuristics
from src.common.parser import Parser
from src.common.problem import CVRPProblem

# Our custom evolutionary engine
from src.ga.multimodal_genetic_algorithm import MultimodalGeneticAlgorithm

: 

## Chromosome Decoding & Local Search Optimization

Unlike ACO, which probabilistically constructs routes step-by-step, our Genetic Algorithm operates on a "giant tour" permutation—a single, continuous sequence of all clients without depot markers. To evaluate these permutations against the strict physical constraints of the CVRP (vehicle capacity), we employ a deterministic decoding strategy. Furthermore, to ensure geometrical efficiency and competitiveness, we elevate our baseline GA into a **Memetic Algorithm** by hybridizing the evolutionary process with an aggressive local search operator.

### Giant Tour Decoding & Implicit Feasibility

Dealing with capacity limits is a major challenge in evolutionary computation. If crossover operators had to constantly monitor vehicle capacities, the generation of valid offspring would be mathematically restrictive and highly inefficient. Instead, we implemented an **Implicit Feasibility (Split) Strategy**:

* **Dynamic Depot Insertion (Hard Constraints):** Our genetic operators (OX1 Crossover and Swap Mutation) only manage the relative ordering of clients, ignoring vehicle capacity entirely. During the decoding phase, the algorithm sequentially reads the permutation, accumulating customer demand. The moment adding the next customer would exceed the maximum vehicle capacity, the algorithm dynamically inserts a return trip to the central depot, resets the load, and dispatches a new vehicle. 
* This elegantly guarantees that **100% of the evaluated solutions are valid** and strictly adhere to capacity limits, completely bypassing the need for complex, heavy penalty functions (Soft Constraints) that can distort the evolutionary fitness landscape.

### Memetic Refinement: Windowed 2-Opt Local Search

While pure evolutionary operators are excellent at identifying promising global regions (exploration), they often struggle with local geometric inefficiencies, frequently leaving "knots" or crossed paths within individual vehicle routes. 

To bridge this gap, our engine applies a **Windowed 2-opt Local Search** to a percentage of the newly generated offspring. This memetic operator scans localized segments of the chromosome, systematically reversing subsets of nodes to untangle crossed edges. By restricting the 2-opt search to a sliding window rather than the entire route, we avoid extreme computational overhead ($O(N^2)$), achieving a highly efficient balance between global evolutionary exploration and localized exploitation.

In [ ]:
# Initialize the environment pointing to the common datasets folder
data_path = './data/X-n106-k14.vrp'  # Change to loop through 'data' folder if executing multiple

print(f"Loading data file: {data_path} ...")
vrp_parser = Parser(data_path)
nodes, demands, capacity = vrp_parser.parse()

# Instantiate the shared problem state
cvrp_instance = CVRPProblem(nodes, demands, capacity)

print(f"Environment ready. Successfully loaded CVRP instance: {len(nodes)} nodes, Vehicle Capacity: {capacity}")

In [ ]:
# Select a small instance for visual demonstration
import os
import matplotlib.pyplot as plt

from src.common.parser import Parser
from src.common.problem import CVRPProblem

sample_instance: str = 'X-n106-k14.vrp'
# Assuming the notebook is at the root or 'experiments' folder
file_path: str = os.path.join('./data', sample_instance) 

# Parse the topology of the terrain
sample_parser: Parser = Parser(file_path)
nodes, demands, capacity = sample_parser.parse()
sample_problem: CVRPProblem = CVRPProblem(nodes, demands, capacity)

# Separate the coordinates for plotting
depot_id: int = sample_problem.depot_id
depot_coord = nodes[depot_id]

# Extract customers' coordinates
customers_x = [coords[0] for node_id, coords in nodes.items() if node_id != depot_id]
customers_y = [coords[1] for node_id, coords in nodes.items() if node_id != depot_id]

# Draw the map
plt.figure(figsize=(8, 6))
# Try to use seaborn style if available, otherwise default
try:
    plt.style.use('seaborn-v0_8-darkgrid')
except:
    pass

# Plot the customers and depot
plt.scatter(customers_x, customers_y, c='blue', s=30, alpha=0.6, edgecolors='k', label='Customers')
plt.scatter(depot_coord[0], depot_coord[1], c='red', marker='s', s=120, edgecolors='k', label='Depot Central', zorder=5)

plt.title(f"CVRP Topology: {sample_instance}", fontweight='bold')
plt.xlabel("Coordinate X")
plt.ylabel("Coordinate Y")
plt.legend()
plt.show()

In [ ]:
# --- EXPERIMENT: Pure GA vs Memetic GA ---
import time

# Hiperparámetros para la demostración visual
DEMO_POP_SIZE = 50
DEMO_GENERATIONS = 150
DEMO_MUTATION = 0.1

print('--- Testing Raw Evolutionary Search (No 2-opt) ---')
ga_pure = MultimodalGeneticAlgorithm(cvrp_instance, pop_size=DEMO_POP_SIZE)

start_time = time.time()
# use_local_search=False por defecto
top3_pure, hist_pure = ga_pure.run(generations=DEMO_GENERATIONS, mutation_rate=DEMO_MUTATION)
pure_time = time.time() - start_time
best_pure_route, best_pure_cost = top3_pure[0]


print('\n--- Testing Enhanced Memetic Search (Windowed 2-opt) ---')
# CAMBIO AQUÍ: Usamos cvrp_instance
ga_memetic = MultimodalGeneticAlgorithm(cvrp_instance, pop_size=DEMO_POP_SIZE)

start_time = time.time()
# Activamos el interruptor memético
top3_memetic, hist_memetic = ga_memetic.run(
    generations=DEMO_GENERATIONS, 
    mutation_rate=DEMO_MUTATION, 
    use_local_search=True
)
memetic_time = time.time() - start_time
best_memetic_route, best_memetic_cost = top3_memetic[0]

print("\n--- PERFORMANCE SUMMARY ---")
print(f"Pure GA Cost: {best_pure_cost:.2f} (Time: {pure_time:.2f}s)")
print(f"Memetic GA Cost: {best_memetic_cost:.2f} (Time: {memetic_time:.2f}s)")
print(f"Improvement: {best_pure_cost - best_memetic_cost:.2f} distance units!")

In [ ]:
# --- VISUALIZATION: The impact of Local Search ---
import numpy as np

def plot_cvrp_route(ax, route, problem, title):
    """Auxiliary function to draw a CVRP route with pastel colors per vehicle."""
    depot_x, depot_y = problem.nodes[problem.depot_id]
    
    # Split the giant tour into individual vehicle routes
    vehicle_routes = []
    current_route = []
    for node in route:
        if node == problem.depot_id:
            if current_route:
                vehicle_routes.append(current_route)
                current_route = []
        else:
            current_route.append(node)
    if current_route:
        vehicle_routes.append(current_route)

    # Use a distinct color map for the vehicles
    colors = plt.cm.tab20(np.linspace(0, 1, len(vehicle_routes)))
    
    for i, sub_route in enumerate(vehicle_routes):
        # Add depot at start and end to close the loop
        full_path = [problem.depot_id] + sub_route + [problem.depot_id]
        xs = [problem.nodes[n][0] for n in full_path]
        ys = [problem.nodes[n][1] for n in full_path]
        
        ax.plot(xs, ys, marker='o', markersize=4, color=colors[i], alpha=0.7, label=f'Vehicle {i+1}')
        
    # Draw the central depot
    ax.plot(depot_x, depot_y, marker='s', color='red', markersize=10, label='Central Depot', zorder=5)
    
    ax.set_title(title, fontweight='bold')
    ax.set_xlabel('Coordinate X')
    ax.set_ylabel('Coordinate Y')
    ax.set_facecolor('#eaeaf2')
    ax.grid(color='white', linestyle='-', linewidth=1)

# Create a dual-plot figure
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(10, 14))

# Top Plot: Pure GA
# CAMBIO AQUÍ: Usamos cvrp_instance en lugar de sample_problem
plot_cvrp_route(ax1, best_pure_route, cvrp_instance, 
                f'Raw Evolutionary Decoding (No 2-opt applied: {best_pure_cost:.2f})')

# Bottom Plot: Memetic GA
# CAMBIO AQUÍ: Usamos cvrp_instance en lugar de sample_problem
plot_cvrp_route(ax2, best_memetic_route, cvrp_instance, 
                f'Enhanced Memetic Decoding (Windowed 2-opt applied: {best_memetic_cost:.2f})')

# Add a single legend outside the plots
handles, labels = ax2.get_legend_handles_labels()
fig.legend(handles, labels, loc='center right', bbox_to_anchor=(1.15, 0.5), frameon=False)

plt.tight_layout()
plt.show()

## Multimodal Techniques: Overcoming Genetic Drift

The probabilistic nature of a Genetic Algorithm naturally generates a high degree of structural diversity during the initial generations. As the completely random initial population begins to cross over and mutate, it explores a vast amount of the solution space.

However, this initial diversity is fleeting. A standard GA is driven by intense selection pressure: the fittest individuals (shortest routes) are exponentially more likely to be selected as parents. Without explicit niching rules, this leads to a phenomenon known as **Genetic Drift** or **Premature Convergence**. The genetic material of the global best solution quickly dominates the entire gene pool, effectively wiping out alternative routing strategies and forcing the entire population to converge into a single, unimodal consensus path.

To systematically guarantee the discovery and preservation of multiple routing alternatives, we must introduce explicit Multimodal approaches. Instead of relying on standard generational replacement, we actively alter the algorithm's survival mechanics to force the population to deliberately divide its search efforts, conquer new geographical zones, and stably maintain multiple high-quality niches simultaneously.

### Deterministic Crowding (Active Niching)

To achieve multimodality, we implemented a highly aggressive replacement strategy known as **Deterministic Crowding**, driven by **Jaccard Distance** edge comparisons. 

Unlike passive archiving (which simply filters solutions at the very end), our GA enforces diversity *during* the evolutionary process. When a new offspring is generated, it does not simply replace the weakest global individual. Instead, its topology (edges) is compared against the existing population. If the offspring is structurally too similar to an existing solution (falling below our `similarity_threshold`), they are forced to compete for the exact same "niche." 

This acts as a hard mathematical "wall" against genetic homogenization, ensuring that the final population is naturally clustered into distinct, high-quality topographical niches. Let's execute the Memetic GA and extract our Top 3 diverse routing alternatives:

In [ ]:
# --- EXPERIMENT: Multimodal Niche Extraction ---
import time
from typing import List, Tuple

# Hyperparameters targeting deep multimodal exploration
MULTIMODAL_POP_SIZE: int = 100
MULTIMODAL_GENERATIONS: int = 200
MULTIMODAL_MUTATION: float = 0.1
SIMILARITY_THRESHOLD: float = 0.20  # Require at least 20% topological difference between niches

print(f"--- Launching Multimodal Memetic GA (Threshold: {SIMILARITY_THRESHOLD}) ---")
# Instantiate the engine using our shared CVRP instance
ga_multimodal = MultimodalGeneticAlgorithm(cvrp_instance, pop_size=MULTIMODAL_POP_SIZE)

start_time_multi: float = time.time()

# Execute the algorithm (Memetic + Crowding)
top3_niches: List[Tuple[List[int], float]]
history_multi: List[float]

top3_niches, history_multi = ga_multimodal.run(
    generations=MULTIMODAL_GENERATIONS, 
    mutation_rate=MULTIMODAL_MUTATION, 
    similarity_threshold=SIMILARITY_THRESHOLD,
    use_local_search=True # Utilizing our Windowed 2-opt for geometric efficiency
)

execution_time_multi: float = time.time() - start_time_multi

print("\n--- MULTIMODAL EXTRACTION COMPLETE ---")
print(f"Execution Time: {execution_time_multi:.2f} seconds")
for i, (route, cost) in enumerate(top3_niches):
    validity: str = "Valid" if cvrp_instance.is_route_valid(route) else "INVALID (Capacity Overflow)"
    print(f"🏆 Niche {i+1} | Cost: {cost:.2f} | Status: {validity}")

In [ ]:
# --- VISUALIZATION: The 3 Distinct Topological Niches ---
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.axes import Axes

def plot_single_niche(ax: Axes, route: List[int], problem: CVRPProblem, title: str) -> None:
    """
    Plots a single CVRP route on the provided matplotlib Axes.
    Strictly type-hinted for production-grade code quality.
    """
    depot_x: float
    depot_y: float
    depot_x, depot_y = problem.nodes[problem.depot_id]
    
    # Segment the giant tour into independent vehicle trajectories
    vehicle_routes: List[List[int]] = []
    current_route: List[int] = []
    
    for node in route:
        if node == problem.depot_id:
            if current_route:
                vehicle_routes.append(current_route)
                current_route = []
        else:
            current_route.append(node)
            
    if current_route:
        vehicle_routes.append(current_route)

    # Dynamic color mapping for varying fleet sizes
    colors = plt.cm.tab20(np.linspace(0, 1, len(vehicle_routes)))
    
    for i, sub_route in enumerate(vehicle_routes):
        # Close the loop by prepending and appending the depot
        full_path: List[int] = [problem.depot_id] + sub_route + [problem.depot_id]
        
        xs: List[float] = [problem.nodes[n][0] for n in full_path]
        ys: List[float] = [problem.nodes[n][1] for n in full_path]
        
        ax.plot(xs, ys, marker='o', markersize=3, color=colors[i], alpha=0.7)
        
    # Overlay the central depot
    ax.plot(depot_x, depot_y, marker='s', color='red', markersize=8, zorder=5)
    
    # Axis styling
    ax.set_title(title, fontsize=10, fontweight='bold')
    ax.set_facecolor('#eaeaf2')
    ax.grid(color='white', linestyle='-', linewidth=0.5)
    ax.set_xticks([]) # Remove ticks for cleaner visual presentation
    ax.set_yticks([])

# Ensure we have at least 3 routes to plot, otherwise pad with empty
while len(top3_niches) < 3:
    top3_niches.append(([], 0.0))

# Initialize a 1x3 horizontal grid
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle(f"Top 3 Structurally Distinct Routing Alternatives (Jaccard Threshold: {SIMILARITY_THRESHOLD * 100}%)", 
             fontsize=14, fontweight='bold', y=1.05)

# Render each niche
for i, ax in enumerate(axes):
    if top3_niches[i][0]: # If route exists
        route, cost = top3_niches[i]
        plot_single_niche(ax, route, cvrp_instance, f"Niche {i+1} Topology\nTotal Cost: {cost:.2f}")
    else:
        ax.set_title(f"Niche {i+1} (Not Found)")
        ax.axis('off')

plt.tight_layout()
plt.show()

## Evolutionary Convergence & Statistical Analysis

To evaluate the stability and learning rate of our Memetic Algorithm, we first analyze its convergence history. A steep initial drop indicates rapid global exploration, while a plateau in later generations suggests that the algorithm has successfully fine-tuned the solutions within their respective niches (exploitation).

Furthermore, unlike algorithms that return a massive unfiltered archive of similar solutions, our Deterministic Crowding approach distills the population into exactly 3 structurally distinct, highly optimized niches. Below, we visualize the evolutionary learning curve alongside the fitness distribution of our final multimodal alternatives.

In [ ]:
# --- ANALYTICS: Convergence and Niche Distribution ---
import matplotlib.pyplot as plt
import numpy as np
from typing import List, Tuple

def plot_ga_analytics(history: List[float], niches: List[Tuple[List[int], float]]) -> None:
    """Renders the convergence curve and statistical distribution of the found niches."""
    
    # 1. Convergence Curve
    plt.figure(figsize=(12, 4))
    plt.plot(history, color='forestgreen', linewidth=2, linestyle='-')
    plt.title("Memetic GA: Convergence History", fontweight='bold')
    plt.xlabel("Generation (Iteration)")
    plt.ylabel("Best Fitness (Distance)")
    plt.grid(True, linestyle='--', alpha=0.6)
    plt.tight_layout()
    plt.show()

    # 2. Statistical Analysis of Final Niches
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
    fig.suptitle("Statistical Analysis of Extracted Multimodal Niches", fontweight='bold')
    
    niche_costs: List[float] = [cost for _, cost in niches if cost > 0]
    labels: List[str] = [f"Niche {i+1}" for i in range(len(niche_costs))]
    
    # Left: Bar chart of Costs
    bars = ax1.bar(labels, niche_costs, color=['#5cb85c', '#5bc0de', '#f0ad4e'], edgecolor='black')
    ax1.set_title("Fitness by Niche")
    ax1.set_ylabel("Route Cost")
    ax1.set_ylim(min(niche_costs) * 0.95, max(niche_costs) * 1.05) # Zoom in for detail
    
    # Right: Boxplot to show dispersion context
    # We use a boxplot to align with the ACO team's visual style
    ax2.boxplot(niche_costs, vert=False, patch_artist=True, 
                boxprops=dict(facecolor='#d9534f', color='black'),
                medianprops=dict(color='white', linewidth=2))
    ax2.set_title("Fitness Dispersion (Boxplot)")
    ax2.set_xlabel("Route Cost")
    ax2.set_yticks([]) # Hide y-axis labels for boxplot

    plt.tight_layout()
    plt.show()

# Call the analytics plotter using the data from the previous cell
plot_ga_analytics(history_multi, top3_niches)

In [ ]:
# --- ANALYTICS: Gene Pool Consensus Heatmap ---
import matplotlib.pyplot as plt
import numpy as np

def plot_gene_pool_heatmap(problem: 'CVRPProblem', niches: List[Tuple[List[int], float]]) -> None:
    """
    Generates a heatmap of edge frequencies (Gene Consensus) across the final niches.
    Acts as the GA counterpart to the ACO Pheromone Matrix.
    """
    num_nodes: int = len(problem.nodes)
    consensus_matrix: np.ndarray = np.zeros((num_nodes, num_nodes))
    
    # Tally the edges used in all extracted niches
    valid_niches = [route for route, cost in niches if route]
    
    for route in valid_niches:
        # Reconstruct full paths with depots
        vehicle_routes = []
        current = []
        for node in route:
            if node == problem.depot_id:
                if current: vehicle_routes.append(current)
                current = []
            else:
                current.append(node)
        if current: vehicle_routes.append(current)
            
        for sub_route in vehicle_routes:
            full_path = [problem.depot_id] + sub_route + [problem.depot_id]
            for i in range(len(full_path) - 1):
                u, v = full_path[i], full_path[i+1]
                # Increment both directions for an undirected interpretation
                consensus_matrix[u][v] += 1
                consensus_matrix[v][u] += 1
                
    # Plotting
    plt.figure(figsize=(8, 6))
    
    # We use the 'magma' colormap to simulate the glowing dark-background style of ACO
    im = plt.imshow(consensus_matrix, cmap='magma', interpolation='nearest', aspect='auto')
    
    plt.colorbar(im, label='Frequency of Edge Occurrence (Consensus Level)')
    plt.title("Gene Pool Consensus Matrix (Edge Frequencies)", fontweight='bold', color='white')
    
    # Styling for a "dark mode" heatmap
    ax = plt.gca()
    ax.set_facecolor('black')
    fig = plt.gcf()
    fig.patch.set_facecolor('#2b2b2b') # Dark grey background
    ax.tick_params(colors='white')
    ax.xaxis.label.set_color('white')
    ax.yaxis.label.set_color('white')
    
    plt.xlabel("From Node ID")
    plt.ylabel("To Node ID")
    
    plt.tight_layout()
    plt.show()

# Render the Heatmap
plot_gene_pool_heatmap(cvrp_instance, top3_niches)

## Mathematical Validation of Structural Diversity

While visual inspection confirms that the niches explore different regions of the map, academic rigor requires mathematical validation. To quantify the structural differences between our extracted solutions, we calculate the **Jaccard Distance** between the edge sets of each route.

A Jaccard distance of 0% implies identical routes, while 100% implies completely disjoint paths with zero shared street connections. Because we configured our Deterministic Crowding threshold to `0.20`, we expect all pairwise comparisons between our top 3 niches to exceed at least a 20% structural difference.

In [ ]:
# --- DIVERSITY METRICS: Mathematical Validation ---
from src.common.diversity import DiversityHandler

print('--- Jaccard Structural Diversity (Deterministic Crowding) ---')

# Extract the raw routes of the 3 niches discovered
route_1: list = top3_niches[0][0] if len(top3_niches) > 0 else []
route_2: list = top3_niches[1][0] if len(top3_niches) > 1 else []
route_3: list = top3_niches[2][0] if len(top3_niches) > 2 else []

if route_1 and route_2 and route_3:
    # Calculate the Jaccard distance crossing the 3 niches using the shared common method
    jaccard_1_2: float = DiversityHandler.calculate_jaccard_distance(route_1, route_2)
    jaccard_1_3: float = DiversityHandler.calculate_jaccard_distance(route_1, route_3)
    jaccard_2_3: float = DiversityHandler.calculate_jaccard_distance(route_2, route_3)

    # Print the percentage results
    print(f'Niche 1 vs Niche 2: {jaccard_1_2:.2%}')
    print(f'Niche 1 vs Niche 3: {jaccard_1_3:.2%}')
    print(f'Niche 2 vs Niche 3: {jaccard_2_3:.2%}')
    
    print(f'\n(Verification: Minimum required diversity threshold was {SIMILARITY_THRESHOLD:.2%})')
else:
    print("Warning: Could not find 3 complete niches to compare.")

## Memetic GA Statistical Rigor (35 Runs)

Due to the inherent stochastic nature of Evolutionary Computation (random initialization, probabilistic crossover, and mutation), a single execution is insufficient to evaluate the true performance of the algorithm. 

To satisfy the evaluation criteria and prove the robustness of our Multimodal Memetic GA configuration, we will execute the algorithm 35 independent times. We will log the execution time and the best fitness found in each run into a Pandas DataFrame. This guarantees a statistically significant sample size ($N \ge 30$) required for the final hypothesis testing and fair benchmarking against the Swarm Intelligence models.

In [ ]:
# --- STATISTICAL RIGOR: 35 Independent Runs ---
import pandas as pd
import numpy as np
import time
from typing import List, Dict, Any

# Test parameters
N_REPEATS: int = 35
STAT_POP_SIZE: int = 100
STAT_GENERATIONS: int = 200
STAT_MUTATION: float = 0.1
STAT_THRESHOLD: float = 0.20

print(f'--- Running {N_REPEATS} repetitions of Multimodal Memetic GA ---')

results_log: List[Dict[str, Any]] = []

# Sweep of 35 iterations
for i in range(N_REPEATS):
    print(f"Executing run {i+1}/{N_REPEATS}...", end='\r')
    
    # Initialize a fresh engine for each run to avoid data leakage
    ga_stat_engine = MultimodalGeneticAlgorithm(cvrp_instance, pop_size=STAT_POP_SIZE)
    
    start_time_run: float = time.time()
    
    # Run the Memetic version
    top3, _ = ga_stat_engine.run(
        generations=STAT_GENERATIONS, 
        mutation_rate=STAT_MUTATION,
        similarity_threshold=STAT_THRESHOLD,
        use_local_search=True
    )
    
    run_time: float = time.time() - start_time_run
    
    # Extract the absolute best fitness from this specific run (Niche 1)
    best_run_cost: float = top3[0][1] if top3 else float('inf')
    
    # Log the data
    results_log.append({
        'run_id': i + 1,
        'best_fitness': best_run_cost,
        'execution_time': run_time
    })

print(f"Execution of {N_REPEATS} runs completed successfully.        ")

# Convert the log into a Pandas DataFrame for easy statistical extraction
df_memetic_ga: pd.DataFrame = pd.DataFrame(results_log)

# Extract basic statistics
mean_fitness: float = df_memetic_ga['best_fitness'].mean()
std_fitness: float = df_memetic_ga['best_fitness'].std()
mean_time: float = df_memetic_ga['execution_time'].mean()
best_overall: float = df_memetic_ga['best_fitness'].min()

# Print a quick summary to the console (matching the team's format)
print('\n--- Memetic GA Performance Summary (35 runs) ---')
print(f"Best Absolute Fitness: {best_overall:.2f}")
print(f"Mean Fitness: {mean_fitness:.2f} ± {std_fitness:.2f}")
print(f"Mean Runtime: {mean_time:.2f} seconds")

# Optional: Save to CSV so you don't lose the data if the notebook restarts!
df_memetic_ga.to_csv('data/results_ga_memetic_35runs.csv', index=False)

## The Archive Method (Passive Niching & Soft Constraints)

Hard constraints (like our Implicit Feasibility Giant Tour split) guarantee that 100% of the evaluated offspring are valid. However, this strictness can sometimes trap the evolutionary search in local optima, as it prevents the algorithm from temporarily exploring slightly invalid topologies that might act as "stepping stones" to a better global solution. 

To parallel the ACO team's approach, we introduce the concept of **Soft Constraints (Penalty Functions)** combined with **Passive Archiving**. 

Instead of actively forcing diversity during the replacement phase (Active Crowding), a Passive Archive method allows the Genetic Algorithm to evolve naturally (and even converge prematurely). However, a background observer silently monitors every single generation, collecting any evaluated route that falls within a competitive fitness margin. At the end of the execution, this massive archive of historical routes is filtered using the Jaccard Distance to extract structurally diverse niches.

In [ ]:
import inspyred
import time
from typing import List, Tuple
from src.common.diversity import DiversityHandler

# Alert message
print('--- Launching Passive Archiving GA (No Crowding) ---')

# We use OOP Inheritance to modify our engine's behavior without touching the core file
class PassiveArchiveGA(MultimodalGeneticAlgorithm):
    def __init__(self, problem, pop_size=100):
        super().__init__(problem, pop_size)
        self.passive_archive: List[Tuple[List[int], float]] = []
        
    def observer_tracker(self, population: list, num_generations: int, num_evaluations: int, args: dict) -> None:
        """Overrides the observer to passively collect good solutions into an archive."""
        # Standard tracking
        best_fitness = min([ind.fitness for ind in population])
        self.cost_history.append(best_fitness)
        
        # Archive collection: Save solutions that are within 10% of the current best
        threshold_fitness = best_fitness * 1.10 
        for ind in population:
            if ind.fitness <= threshold_fitness:
                route = self.decode_chromosome(ind.candidate)
                self.passive_archive.append((route, ind.fitness))
                
        if num_generations % 20 == 0:
            print(f"Gen {num_generations:3d} | Best: {best_fitness:.2f} | Archive Size: {len(self.passive_archive)}")

    def run_passive(self, generations: int = 150, mutation_rate: float = 0.1, similarity_threshold: float = 0.20) -> tuple:
        """Executes a standard GA and filters the passive archive at the end."""
        prng = random.Random()
        ga_engine = inspyred.ec.EvolutionaryComputation(prng)

        ga_engine.selector = inspyred.ec.selectors.tournament_selection
        # CRITICAL CHANGE: We use standard generational replacement (NO CROWDING/ACTIVE NICHING)
        ga_engine.replacer = inspyred.ec.replacers.generational_replacement 
        ga_engine.variator = self.custom_variator
        ga_engine.terminator = inspyred.ec.terminators.generation_termination
        ga_engine.observer = self.observer_tracker

        self.cost_history = []
        self.passive_archive = []

        ga_engine.evolve(
            generator=self.generate_chromosome, evaluator=self.evaluate_population, pop_size=self.pop_size,
            bounder=inspyred.ec.DiscreteBounder(self.clients), maximize=False, max_generations=generations,
            tournament_size=3, mutation_rate=mutation_rate, use_local_search=True
        )

        # Post-processing: Filter the massive archive using Jaccard Distance
        self.passive_archive.sort(key=lambda x: x[1]) # Sort by fitness
        filtered_niches = []
        
        for route, cost in self.passive_archive:
            if len(filtered_niches) >= 3:
                break
            is_novel = True
            for accepted_route, _ in filtered_niches:
                if DiversityHandler.calculate_jaccard_distance(route, accepted_route) < similarity_threshold:
                    is_novel = False
                    break
            if is_novel:
                filtered_niches.append((route, cost))

        return filtered_niches, self.cost_history

# --- Execute the Experiment ---
ga_passive = PassiveArchiveGA(cvrp_instance, pop_size=100)

start_time_pass = time.time()
alts_pass_arch, history_pass = ga_passive.run_passive(
    generations=200, 
    mutation_rate=0.1, 
    similarity_threshold=0.20
)
time_pass = time.time() - start_time_pass

print(f"\n✅ Passive Archiving Complete in {time_pass:.2f} seconds.")
print(f"Total raw solutions collected in background: {len(ga_passive.passive_archive)}")
print("--- Extracted Niches ---")
for i, (route, cost) in enumerate(alts_pass_arch):
    print(f"Niche {i+1} | Cost: {cost:.2f}")

In [ ]:
# --- VISUALIZATION: Passive Archive Evolution & Statistics ---
import matplotlib.pyplot as plt
import numpy as np

print("Here we show the evolution of the GA along with the corresponding statistical plots of the generated archive.")

# 1. Plot the Evolution (Matching the ACO style)
plt.figure(figsize=(10, 5))

# Plotting the best fitness history (Solid Green Line)
plt.plot(history_pass, color='#2ca02c', linewidth=2, label='Best Fitness')

# Styling to match the ACO visualizer's dark/grey theme
plt.title('Fitness over Generations (Passive GA)', fontweight='bold', fontsize=14)
plt.xlabel('Generation (Iteration)')
plt.ylabel('Fitness (Distance)')
plt.grid(color='white', linestyle='-', linewidth=1)
plt.gca().set_facecolor('#eaeaf2') # Light grey background
plt.legend()
plt.tight_layout()
plt.show()


# 2. Extract the costs from the final alternatives to analyze the distribution
archive_costs_pass: list = [cost for _, cost in alts_pass_arch]

# 3. Draw the statistical graphics (Histogram & Boxplot)
if archive_costs_pass:
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
    fig.suptitle("Statistical Analysis of Final Passive Archive", fontweight='bold')
    
    # Left: Histogram
    ax1.hist(archive_costs_pass, bins=min(10, len(archive_costs_pass)), color='#5cb85c', edgecolor='black', alpha=0.8)
    ax1.set_title("Fitness Distribution (Histogram)")
    ax1.set_xlabel("Route Cost")
    ax1.set_ylabel("Number of Solutions")
    ax1.set_facecolor('#eaeaf2')
    ax1.grid(color='white', linestyle='-', linewidth=0.5)
    
    # Right: Boxplot
    ax2.boxplot(archive_costs_pass, vert=False, patch_artist=True, 
                boxprops=dict(facecolor='#d9534f', color='black'),
                medianprops=dict(color='white', linewidth=2))
    ax2.set_title("Fitness Dispersion (Boxplot)")
    ax2.set_xlabel("Route Cost")
    ax2.set_yticks([]) # Hide y-axis labels
    ax2.set_facecolor('#eaeaf2')
    ax2.grid(color='white', linestyle='-', linewidth=0.5)
    
    plt.tight_layout()
    plt.show()
else:
    print("Archive is empty. No statistical distribution to show.")

## Passive Archive Gene Pool Consensus

To further understand the evolutionary learning process of our Passive Archiving strategy, we can visualize the total accumulated knowledge of the population. Just as an ant colony reinforces pheromone trails, a Genetic Algorithm implicitly reinforces specific edge transitions by selecting and recombining high-quality parent chromosomes.

The following heatmap displays the **Relative Edge Frequency** across *all* the competitive solutions collected in our massive passive archive. The dark background represents unexplored paths, while the bright spots indicate the specific "highways" that the algorithm consistently relied upon to build high-fitness routes, long before we applied the final Jaccard distance filter.

In [ ]:
# --- VISUALIZATION: Passive Archive Gene Consensus Heatmap ---
import matplotlib.pyplot as plt
import numpy as np

print('--- Gene Pool Consensus Heatmap (Passive Archive) ---')

def plot_massive_archive_heatmap(problem: 'CVRPProblem', archive: list) -> None:
    """
    Plots a heatmap showing the frequency of node-to-node transitions 
    across all solutions stored in the passive archive.
    """
    num_nodes: int = len(problem.nodes)
    consensus_matrix: np.ndarray = np.zeros((num_nodes, num_nodes))
    
    # Iterate over all routes found in the massive passive archive
    for route, _ in archive:
        if not route: 
            continue
        
        # Extract vehicle subroutes 
        vehicle_routes = []
        current = []
        for node in route:
            if node == problem.depot_id:
                if current: 
                    vehicle_routes.append(current)
                current = []
            else:
                current.append(node)
        if current: 
            vehicle_routes.append(current)
            
        for sub_route in vehicle_routes:
            full_path = [problem.depot_id] + sub_route + [problem.depot_id]
            for i in range(len(full_path) - 1):
                u, v = full_path[i], full_path[i+1]
                # Increment both directions (Undirected graph approach)
                consensus_matrix[u][v] += 1
                consensus_matrix[v][u] += 1  
                
    # Normalize the matrix to values between 0 and 1 (like pheromone concentrations)
    max_val = np.max(consensus_matrix)
    if max_val > 0:
        consensus_matrix = consensus_matrix / max_val
        
    plt.figure(figsize=(10, 8))
    
    # Mask absolute zero values so they appear completely dark
    masked_matrix = np.ma.masked_where(consensus_matrix == 0, consensus_matrix)
    
    # Use 'magma' colormap and force the background to be almost black
    cmap = plt.cm.magma
    cmap.set_bad(color='#111116') 
    
    im = plt.imshow(masked_matrix, cmap=cmap, interpolation='nearest', aspect='auto')
    
    # Styling to match the ACO team's dark aesthetic
    cbar = plt.colorbar(im, label='Relative Edge Frequency (Gene Consensus)')
    cbar.ax.yaxis.label.set_color('white')
    cbar.ax.tick_params(colors='white')
    
    plt.title("Gene Pool Consensus Matrix (Node-to-Node Transitions)", fontweight='bold', color='white', pad=15)
    plt.xlabel("Source Node ID", color='white')
    plt.ylabel("Destination Node ID", color='white')
    
    # Dark theme background application
    ax = plt.gca()
    fig = plt.gcf()
    fig.patch.set_facecolor('#1e1e1e') # VSCode dark grey
    ax.set_facecolor('#1e1e1e')
    ax.tick_params(colors='white')
    
    # Add a subtle grid just like the ACO screenshot
    ax.grid(color='white', linestyle='-', linewidth=0.5, alpha=0.3)
    
    plt.tight_layout()
    plt.show()

# Execute the visualization using the passive archive we populated in the previous step
if hasattr(ga_passive, 'passive_archive') and ga_passive.passive_archive:
    plot_massive_archive_heatmap(cvrp_instance, ga_passive.passive_archive)
else:
    print("Error: Passive archive is empty or not found.")

## Validation of Passive Archive Niches

To conclude our exploration of the Passive Archiving method, we must mathematically and visually validate the extracted niches. Just as we did with the Active Crowding method, we calculate the pairwise Jaccard Distances to ensure our post-processing filter successfully isolated structurally diverse routes. Finally, we plot the physical topologies of these alternative solutions side-by-side.

In [ ]:
# --- DIVERSITY METRICS & TOPOLOGY PLOTTING (PASSIVE ARCHIVE) ---
from src.common.diversity import DiversityHandler
import matplotlib.pyplot as plt
import numpy as np

# Console message for diversity
print('--- Jaccard Structural Diversity (Passive Archive Method) ---')

# Extract the raw routes of the 3 niches discovered
route_1_pass: list = alts_pass_arch[0][0] if len(alts_pass_arch) > 0 else []
route_2_pass: list = alts_pass_arch[1][0] if len(alts_pass_arch) > 1 else []
route_3_pass: list = alts_pass_arch[2][0] if len(alts_pass_arch) > 2 else []

if route_1_pass and route_2_pass and route_3_pass:
    # Calculate the Jaccard distance crossing the 3 niches
    jaccard_1_2_pass: float = DiversityHandler.calculate_jaccard_distance(route_1_pass, route_2_pass)
    jaccard_1_3_pass: float = DiversityHandler.calculate_jaccard_distance(route_1_pass, route_3_pass)
    jaccard_2_3_pass: float = DiversityHandler.calculate_jaccard_distance(route_2_pass, route_3_pass)

    # Print the percentage results
    print(f'Niche 1 vs Niche 2: {jaccard_1_2_pass:.2%}')
    print(f'Niche 1 vs Niche 3: {jaccard_1_3_pass:.2%}')
    print(f'Niche 2 vs Niche 3: {jaccard_2_3_pass:.2%}\n')
else:
    print("Warning: Could not find 3 complete niches to compare.\n")

# Prepare the canvas with 3 horizontal subplots
fig, axs = plt.subplots(1, 3, figsize=(18, 5))

# Iterate through each plot and alternative
for i, ax in enumerate(axs):
    if i < len(alts_pass_arch):
        # Evaluate the real distance of the route
        ruta_pasiva, fitness_limpio = alts_pass_arch[i]
        
        # Assign the title with the exact fitness
        ax.set_title(f'Archive Niche (Passive) #{i+1}\nFitness: {fitness_limpio:.2f}', fontweight='bold')
        
        # Draw all customers in light gray as background
        for nid, coords in cvrp_instance.nodes.items():
            if nid != cvrp_instance.depot_id:
                ax.scatter(coords[0], coords[1], c='gray', s=15, alpha=0.5)
                
        # Split the giant tour into individual vehicle routes (GA format translation)
        vehicle_routes = []
        current_route = []
        for node in ruta_pasiva:
            if node == cvrp_instance.depot_id:
                if current_route:
                    vehicle_routes.append(current_route)
                    current_route = []
            else:
                current_route.append(node)
        if current_route:
            vehicle_routes.append(current_route)
            
        colors = plt.cm.tab20(np.linspace(0, 1, len(vehicle_routes)))
        
        # Iterate through each truck's individual trip
        for num_camion, viaje in enumerate(vehicle_routes):
            # Filter if it's a valid trip
            if len(viaje) > 0:
                # Reconstruct the full route adding the depot at start and end
                viaje_completo = [cvrp_instance.depot_id] + viaje + [cvrp_instance.depot_id]
                
                # Extract X and Y coordinates
                cx: list = [cvrp_instance.nodes[n][0] for n in viaje_completo]
                cy: list = [cvrp_instance.nodes[n][1] for n in viaje_completo]
                
                ax.plot(cx, cy, marker='o', markersize=4, color=colors[num_camion], alpha=0.8)
        
        # Draw the central depot
        depot_x, depot_y = cvrp_instance.nodes[cvrp_instance.depot_id]
        ax.plot(depot_x, depot_y, marker='s', color='red', markersize=10, zorder=5)
        
        # Styling
        ax.set_facecolor('#eaeaf2')
        ax.grid(color='white', linestyle='-', linewidth=0.5)
        ax.set_xticks([]) # Clean look without axis numbers
        ax.set_yticks([])
    else:
        ax.set_title(f"Archive Niche #{i+1}\n(Not Found)")
        ax.axis('off')

plt.tight_layout()
plt.show()

## Passive Archive Statistical Rigor (35 Runs)

To satisfy the evaluation criteria and allow for a fair, rigorous comparison against both our Active Crowding method and the Swarm Intelligence models (ACO & PSO), we must evaluate the stability of the Passive Archiving configuration. 

We will run this specific configuration 35 independent times and log the execution times and best fitness values into a Pandas DataFrame. This ensures we have a statistically significant sample for the final hypothesis testing phase.

In [ ]:
# --- STATISTICAL RIGOR: 35 Independent Runs (Passive Archive) ---
import pandas as pd
import numpy as np
import time

print(f'--- Running {N_REPEATS} repetitions of Passive Archive GA ---')

results_log_pass: list = []

# Execute the sweep of 35 iterations and save to a dataframe
for i in range(N_REPEATS):
    print(f"Processing GA experiment (Passive): X-n106-k14.vrp - run {i+1}...", end='\r')
    
    # Initialize a fresh passive engine for each run
    ga_stat_passive = PassiveArchiveGA(cvrp_instance, pop_size=STAT_POP_SIZE)
    
    start_time_run = time.time()
    
    # Run the passive version
    niches_pass, _ = ga_stat_passive.run_passive(
        generations=STAT_GENERATIONS, 
        mutation_rate=STAT_MUTATION,
        similarity_threshold=STAT_THRESHOLD
    )
    
    run_time = time.time() - start_time_run
    
    # Extract the absolute best fitness from this specific run
    best_run_cost = niches_pass[0][1] if niches_pass else float('inf')
    
    # Log the data
    results_log_pass.append({
        'run_id': i + 1,
        'best_fitness': best_run_cost,
        'tiempo': run_time  # Using 'tiempo' to match your team's exact DataFrame keys
    })

print(f"Processing GA experiment (Passive): X-n106-k14.vrp - {N_REPEATS} runs completed.        ")

# Convert the log into a Pandas DataFrame
df_pass_arch: pd.DataFrame = pd.DataFrame(results_log_pass)

# Print a quick summary to the console (Matching the ACO/PSO format exactly)
print('\n--- Passive Archive Performance Summary (35 runs) ---')
print(f"Best Absolute Fitness: {df_pass_arch['best_fitness'].min():.2f}")

mean_fit = df_pass_arch['best_fitness'].mean()
std_fit = df_pass_arch['best_fitness'].std()
print(f"Mean Fitness: {mean_fit:.2f} ± {std_fit:.2f}")

mean_time = df_pass_arch['tiempo'].mean()
print(f"Mean Runtime: {mean_time:.2f} seconds")

df_pass_arch.to_csv('data/results_ga_passive_35runs.csv', index=False)

## The Sequential Method (Fitness Derating & Tabu Search)

While Passive Archiving relies on natural exploration, and Crowding forces diversity during replacement, **Sequential Niching** completely alters the fitness landscape dynamically. In this approach, the execution is divided into distinct intervals (epochs). 

When an interval finishes, the best route found is stored as a valid niche and added to a **Tabu List**. Immediately after, a massive "Spatial Fitness Penalty" (Derating) is applied for the next intervals: any new chromosome generated that shares too much topological similarity (Jaccard Distance) with a known Tabu route has its fitness heavily penalized. 

Just like erasing pheromones in ACO, this effectively "poisons" the genetic memory of that specific optimal route, forcing the evolutionary process to completely abandon that geographical zone and converge upon entirely new map regions. We combine this active memory erasure with our implicit **Feasibility** rule (Giant Tour Split) to ensure all explored alternatives remain capacity-compliant.

In [ ]:
from typing import List, Tuple

# Alert message
print('--- Launching Sequential Niching GA with Feasibility Constraints (3 Niches) ---')

class SequentialNichingGA(MultimodalGeneticAlgorithm):
    def __init__(self, problem, pop_size=100):
        super().__init__(problem, pop_size)
        self.tabu_niches: List[List[int]] = [] # Memory of forbidden routes
        self.history_sequential: List[float] = []

    def evaluate_population(self, candidates: list, args: dict) -> list:
        """
        Evaluates fitness but applies a massive penalty to routes too similar 
        to already discovered (Tabu) niches.
        """
        fitness_values = []
        penalty_factor = 10.0 # Multiply cost by 10 to kill it genetically
        similarity_threshold = args.setdefault('similarity_threshold', 0.20)

        for candidate in candidates:
            decoded_route = self.decode_chromosome(candidate)
            base_fitness = self.problem.evaluate_route_distance(decoded_route)

            is_tabu = False
            for tabu_route in self.tabu_niches:
                if DiversityHandler.calculate_jaccard_distance(decoded_route, tabu_route) < similarity_threshold:
                    is_tabu = True
                    break

            if is_tabu:
                fitness_values.append(base_fitness * penalty_factor)
            else:
                fitness_values.append(base_fitness)

        return fitness_values

    def run_sequential(self, num_niches: int=3, generations_per_niche: int=100, mutation_rate: float=0.1, similarity_threshold: float=0.20) -> tuple:
        extracted_niches = []
        
        start_time = time.time()

        for interval in range(num_niches):
            print(f"--> Running Sequential Interval {interval+1}/{num_niches}...")
            
            prng = random.Random()
            ga_engine = inspyred.ec.EvolutionaryComputation(prng)

            ga_engine.selector = inspyred.ec.selectors.tournament_selection
            # We use standard replacement because the penalty function handles the diversity naturally
            ga_engine.replacer = inspyred.ec.replacers.generational_replacement 
            ga_engine.variator = self.custom_variator
            ga_engine.terminator = inspyred.ec.terminators.generation_termination
            
            self.cost_history = []
            ga_engine.observer = self.observer_tracker

            final_pop = ga_engine.evolve(
                generator=self.generate_chromosome,
                evaluator=self.evaluate_population,
                pop_size=self.pop_size,
                bounder=inspyred.ec.DiscreteBounder(self.clients),
                maximize=False,
                max_generations=generations_per_niche,
                tournament_size=3,
                mutation_rate=mutation_rate,
                similarity_threshold=similarity_threshold,
                use_local_search=True
            )

            # Extract the best from this interval
            final_pop.sort(key=lambda x: x.fitness)
            best_ind = final_pop[0]
            best_route = self.decode_chromosome(best_ind.candidate)
            
            # The actual physical distance (ignoring artificial penalties)
            real_cost = self.problem.evaluate_route_distance(best_route)
            
            extracted_niches.append((best_route, real_cost))
            
            # CRITICAL: Add to Tabu list so the next interval avoids this topology
            self.tabu_niches.append(best_route)
            
            # Store history continuously
            self.history_sequential.extend(self.cost_history)
            
        print(f"\nSequential Extraction Complete in {time.time() - start_time:.2f} seconds.")
        return extracted_niches, self.history_sequential


# --- Execute the Experiment ---
ga_seq = SequentialNichingGA(cvrp_instance, pop_size=100)

alts_feas_seq, history_seq = ga_seq.run_sequential(
    num_niches=3,
    generations_per_niche=80, # 80 gens per interval * 3 intervals = 240 gens total
    mutation_rate=0.1, 
    similarity_threshold=0.20
)

print("--- Extracted Niches (Sequential Derating) ---")
for i, (route, cost) in enumerate(alts_feas_seq):
    print(f"Niche {i+1} | Real Cost: {cost:.2f}")

In [ ]:
# --- VISUALIZATION: Sequential Archive Evolution & Statistics ---
import matplotlib.pyplot as plt
import numpy as np

print("Here we show the evolution of the GA along with the corresponding statistical plots of the generated niches.")

# 1. Plot the Evolution (Matching the ACO style)
plt.figure(figsize=(10, 5))

# Plotting the best fitness history (Solid Green Line)
plt.plot(history_seq, color='#2ca02c', linewidth=2, label='Best Fitness')

# Add vertical dashed lines to show exactly where the Tabu Penalty hit
interval_size = 80 # This matches the generations_per_niche we set earlier
num_intervals = 3
for i in range(1, num_intervals):
    plt.axvline(x=i*interval_size, color='red', linestyle='--', alpha=0.8, 
                label='Tabu Penalty Applied (New Niche)' if i == 1 else "")

# Styling to match the ACO visualizer
plt.title('Fitness over Generations (Sequential GA with Tabu Penalty)', fontweight='bold', fontsize=14)
plt.xlabel('Generation (Iteration)')
plt.ylabel('Fitness (Distance / Penalized Cost)')
plt.grid(color='white', linestyle='-', linewidth=1)
plt.gca().set_facecolor('#eaeaf2') # Light grey background
plt.legend()
plt.tight_layout()
plt.show()


# 2. Extract the real costs from the final alternatives to analyze the distribution
archive_costs_seq: list = [cost for _, cost in alts_feas_seq]

# 3. Draw the statistical graphics (Histogram & Boxplot)
if archive_costs_seq:
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
    fig.suptitle("Statistical Analysis of Final Sequential Niches", fontweight='bold')
    
    # Left: Histogram
    # We use fewer bins since we specifically extracted exactly 3 niches
    ax1.hist(archive_costs_seq, bins=3, color='#5cb85c', edgecolor='black', alpha=0.8)
    ax1.set_title("Fitness Distribution (Histogram)")
    ax1.set_xlabel("Route Cost")
    ax1.set_ylabel("Number of Solutions")
    ax1.set_facecolor('#eaeaf2')
    ax1.grid(color='white', linestyle='-', linewidth=0.5)
    
    # Right: Boxplot
    ax2.boxplot(archive_costs_seq, vert=False, patch_artist=True, 
                boxprops=dict(facecolor='#d9534f', color='black'),
                medianprops=dict(color='white', linewidth=2))
    ax2.set_title("Fitness Dispersion (Boxplot)")
    ax2.set_xlabel("Route Cost")
    ax2.set_yticks([]) # Hide y-axis labels
    ax2.set_facecolor('#eaeaf2')
    ax2.grid(color='white', linestyle='-', linewidth=0.5)
    
    plt.tight_layout()
    plt.show()
else:
    print("Archive is empty. No statistical distribution to show.")

In [ ]:
# --- VISUALIZATION: Sequential Niching Gene Consensus Heatmap ---
import matplotlib.pyplot as plt
import numpy as np

print('--- Gene Pool Consensus Heatmap (Sequential Method) ---')

def plot_sequential_heatmap(problem: 'CVRPProblem', niches: list) -> None:
    """
    Plots a heatmap showing the frequency of node-to-node transitions 
    specifically for the disjoint niches extracted via Tabu Search.
    """
    num_nodes: int = len(problem.nodes)
    consensus_matrix: np.ndarray = np.zeros((num_nodes, num_nodes))
    
    # Iterate over the explicitly extracted Sequential (Tabu) routes
    for route, _ in niches:
        if not route: 
            continue
        
        # Extract vehicle subroutes
        vehicle_routes = []
        current = []
        for node in route:
            if node == problem.depot_id:
                if current: 
                    vehicle_routes.append(current)
                current = []
            else:
                current.append(node)
        if current: 
            vehicle_routes.append(current)
            
        for sub_route in vehicle_routes:
            full_path = [problem.depot_id] + sub_route + [problem.depot_id]
            for i in range(len(full_path) - 1):
                u, v = full_path[i], full_path[i+1]
                # Increment both directions (Undirected graph approach)
                consensus_matrix[u][v] += 1
                consensus_matrix[v][u] += 1  
                
    # Normalize the matrix to values between 0 and 1
    max_val = np.max(consensus_matrix)
    if max_val > 0:
        consensus_matrix = consensus_matrix / max_val
        
    plt.figure(figsize=(10, 8))
    
    # Mask absolute zero values so they appear completely dark
    masked_matrix = np.ma.masked_where(consensus_matrix == 0, consensus_matrix)
    
    # Use 'magma' colormap and force the background to be almost black
    cmap = plt.cm.magma
    cmap.set_bad(color='#111116') 
    
    im = plt.imshow(masked_matrix, cmap=cmap, interpolation='nearest', aspect='auto')
    
    # Styling to match the ACO team's dark aesthetic
    cbar = plt.colorbar(im, label='Relative Edge Frequency (Sequential Consensus)')
    cbar.ax.yaxis.label.set_color('white')
    cbar.ax.tick_params(colors='white')
    
    plt.title("Gene Pool Consensus Matrix (Sequential Tabu Niches)", fontweight='bold', color='white', pad=15)
    plt.xlabel("Source Node ID", color='white')
    plt.ylabel("Destination Node ID", color='white')
    
    # Dark theme background application
    ax = plt.gca()
    fig = plt.gcf()
    fig.patch.set_facecolor('#1e1e1e') # VSCode dark grey
    ax.set_facecolor('#1e1e1e')
    ax.tick_params(colors='white')
    
    # Add a subtle grid just like the ACO screenshot
    ax.grid(color='white', linestyle='-', linewidth=0.5, alpha=0.3)
    
    plt.tight_layout()
    plt.show()

# Execute the visualization using the disjoint niches we just extracted
if alts_feas_seq:
    plot_sequential_heatmap(cvrp_instance, alts_feas_seq)
else:
    print("Error: No sequential niches found to plot.")

## Sequential Method Statistical Rigor (35 Runs)

To satisfy the evaluation criteria and rigorously evaluate the consistency of our active memory erasure (Tabu Search) technique, we will run this configuration 35 independent times. We will log the results into a Pandas DataFrame, preparing the data for the final statistical hypothesis testing phase against the Swarm Intelligence models.

In [ ]:
# --- STATISTICAL RIGOR: 35 Independent Runs (Sequential Tabu) ---
import pandas as pd
import numpy as np
import time

print(f'--- Running {N_REPEATS} repetitions of Sequential (Tabu) GA ---')

results_log_seq: list = []

# Execute the sweep of 35 iterations and save to a dataframe
for i in range(N_REPEATS):
    print(f"Processing GA experiment (Sequential): X-n106-k14.vrp - run {i+1}...", end='\r')
    
    # Initialize a fresh sequential engine for each run
    ga_stat_seq = SequentialNichingGA(cvrp_instance, pop_size=STAT_POP_SIZE)
    
    start_time_run = time.time()
    
    # Run the sequential version (3 intervals of 80 generations to match our previous test)
    niches_seq, _ = ga_stat_seq.run_sequential(
        num_niches=3,
        generations_per_niche=80, 
        mutation_rate=STAT_MUTATION,
        similarity_threshold=STAT_THRESHOLD
    )
    
    run_time = time.time() - start_time_run
    
    # Extract the absolute best fitness from this specific run (usually the very first niche found before tabu penalties)
    best_run_cost = niches_seq[0][1] if niches_seq else float('inf')
    
    # Log the data
    results_log_seq.append({
        'run_id': i + 1,
        'best_fitness': best_run_cost,
        'tiempo': run_time  # Using 'tiempo' to match your team's exact DataFrame keys
    })

print(f"Processing GA experiment (Sequential): X-n106-k14.vrp - {N_REPEATS} runs completed.        ")

# Convert the log into a Pandas DataFrame
df_feas_seq: pd.DataFrame = pd.DataFrame(results_log_seq)

# Print a quick summary to the console (Matching the ACO format exactly)
print('\n--- Sequential (Tabu) Performance Summary (35 runs) ---')
print(f"Best Absolute Fitness: {df_feas_seq['best_fitness'].min():.2f}")

mean_fit_seq = df_feas_seq['best_fitness'].mean()
std_fit_seq = df_feas_seq['best_fitness'].std()
print(f"Mean Fitness: {mean_fit_seq:.2f} ± {std_fit_seq:.2f}")

mean_time_seq = df_feas_seq['tiempo'].mean()
print(f"Mean Runtime: {mean_time_seq:.2f} seconds")

# save to CSV for the final comparison notebook
df_feas_seq.to_csv('data/results_ga_sequential_35runs.csv', index=False)

## The Sequential Method (Adaptive Penalty Constraints)

Finally, we test the most aggressive and chaotic configuration by combining our two active disruption techniques: **Soft constraints (adaptive penalties)** and **Sequential niching (Active Memory Erasure / Tabu Search)**.

In this setup, we strip the Genetic Algorithm of its intelligent `Implicit Feasibility` decoder. The chromosomes are now granted absolute spatial freedom during construction—they are allowed to significantly overload the vehicles, creating physically invalid routes. However, during the evaluation phase, any capacity violation incurs a massive mathematical penalty proportional to the excess weight. 

Furthermore, any optimum that the population manages to consolidate under these harsh conditions is stored in the Tabu List, and its topology is actively penalized (erased from the gene pool) in subsequent intervals. This configuration tests whether granting the evolutionary process total exploratory freedom (while simultaneously punishing overloads and forcing amnesia) yields better structural diversity than strictly restricting their physical movements.

In [ ]:
# --- EXPERIMENT: Sequential Niching + Penalty Constraints ---

print('--- Launching sequential niching with penalty constraints (3 niches) ---')

class SequentialPenaltyGA(SequentialNichingGA):
    def decode_chromosome(self, chromosome: list) -> list:
        """
        SOFT CONSTRAINTS: We deliberately break the perfect feasibility.
        We allow the vehicle to overload up to 150% of its capacity before forcing a return.
        """
        route = [self.problem.depot_id]
        current_load = 0

        for client in chromosome:
            demand = self.problem.demands[client]
            # We grant spatial freedom: only force return if it's crazily overloaded (>1.5x)
            if current_load + demand > self.problem.capacity * 1.5:
                route.append(self.problem.depot_id)
                current_load = 0
            route.append(client)
            current_load += demand
        
        if route[-1] != self.problem.depot_id:
            route.append(self.problem.depot_id)
        
        return route

    def evaluate_population(self, candidates: list, args: dict) -> list:
        """
        Calculates fitness applying TWO massive penalties: 
        1. Capacity Overload (Soft constraints)
        2. Tabu Similarity (Sequential niching)
        """
        fitness_values = []
        tabu_penalty_factor = 10.0 # Multiply base cost by 10
        capacity_penalty_weight = 50.0 # Add 50 distance units per unit of excess weight
        similarity_threshold = args.setdefault('similarity_threshold', 0.20)

        for candidate in candidates:
            decoded_route = self.decode_chromosome(candidate)
            base_fitness = self.problem.evaluate_route_distance(decoded_route)

            # --- 1. Penalty for Overloading ---
            overload_penalty = 0
            current_load = 0
            for node in decoded_route:
                if node == self.problem.depot_id:
                    if current_load > self.problem.capacity:
                        overload_penalty += (current_load - self.problem.capacity) * capacity_penalty_weight
                    current_load = 0
                else:
                    current_load += self.problem.demands[node]

            # --- 2. Penalty for Tabu Similarity ---
            is_tabu = False
            for tabu_route in self.tabu_niches:
                if DiversityHandler.calculate_jaccard_distance(decoded_route, tabu_route) < similarity_threshold:
                    is_tabu = True
                    break

            # Calculate final mathematically penalized fitness
            final_fitness = base_fitness + overload_penalty
            if is_tabu:
                final_fitness *= tabu_penalty_factor
                
            fitness_values.append(final_fitness)

        return fitness_values

# --- Execute the final experiment ---
ga_pen_seq = SequentialPenaltyGA(cvrp_instance, pop_size=100)

start_time = time.time()
alts_pen_seq, history_pen_seq = ga_pen_seq.run_sequential(
    num_niches=3,
    generations_per_niche=80, 
    mutation_rate=0.15, # Increased mutation for extra chaos
    similarity_threshold=0.20
)
exec_time = time.time() - start_time

print(f"\nSequential and penalty extraction complete in {exec_time:.2f} seconds.")
print("--- Extracted niches ---")
for i, (route, cost) in enumerate(alts_pen_seq):
    validity = "Valid" if cvrp_instance.is_route_valid(route) else "INVALID (Overloaded)"
    print(f"Niche {i+1} | Real Distance: {cost:.2f} | Status: {validity}")

## Validation of adaptive penalty & sequential niches

Having granted the population total freedom to overload and actively erasing their memory, we now mathematically and visually validate the chaotic outcome. We calculate the pairwise Jaccard Distances to ensure structural diversity. Finally, we plot the physical topologies.

It is crucial to note that, due to our intentional implementation of **Soft Penalty Constraints** in this specific experiment, the extracted routes **might be physically invalid** (vehicles exceeding capacity). If so, we will label them accordingly to distinguish the theoretical cost from the real, geometrically efficient distance.

In [ ]:
# --- DIVERSITY METRICS & TOPOLOGY PLOTTING (PENALTY + SEQUENTIAL) ---
from src.common.diversity import DiversityHandler
import matplotlib.pyplot as plt
import numpy as np

# Console message for diversity
print('--- Jaccard structural diversity of sequential + penalty method ---')

# Extract the raw routes of the 3 niches discovered
route_1_pen: list = alts_pen_seq[0][0] if len(alts_pen_seq) > 0 else []
route_2_pen: list = alts_pen_seq[1][0] if len(alts_pen_seq) > 1 else []
route_3_pen: list = alts_pen_seq[2][0] if len(alts_pen_seq) > 2 else []

if route_1_pen and route_2_pen and route_3_pen:
    # Calculate the Jaccard distance crossing the 3 niches
    jaccard_1_2_pen: float = DiversityHandler.calculate_jaccard_distance(route_1_pen, route_2_pen)
    jaccard_1_3_pen: float = DiversityHandler.calculate_jaccard_distance(route_1_pen, route_3_pen)
    jaccard_2_3_pen: float = DiversityHandler.calculate_jaccard_distance(route_2_pen, route_3_pen)

    # Print the percentage results
    print(f'Niche 1 vs Niche 2: {jaccard_1_2_pen:.2%}')
    print(f'Niche 1 vs Niche 3: {jaccard_1_3_pen:.2%}')
    print(f'Niche 2 vs Niche 3: {jaccard_2_3_pen:.2%}\n')
else:
    print("Warning: Could not find 3 complete penalty niches to compare.\n")

# Prepare the canvas with 3 horizontal subplots
fig, axs = plt.subplots(1, 3, figsize=(18, 5))

# Iterate through each plot and alternative
for i, ax in enumerate(axs):
    if i < len(alts_pen_seq):
        # Evaluate the real distance and its capacity status
        ruta_pen, fitness_real = alts_pen_seq[i]
        is_valid = cvrp_instance.is_route_valid(ruta_pen)
        validity_text = "VALID" if is_valid else "INVALID (Overload)"
        
        # Assign the title with the exact fitness and validity status
        title_color = 'green' if is_valid else 'red'
        ax.set_title(f'Chaos Niche #{i+1}\nDistance: {fitness_real:.2f}\n[{validity_text}]', fontweight='bold', color=title_color)
        
        # Draw all customers in light gray as background
        for nid, coords in cvrp_instance.nodes.items():
            if nid != cvrp_instance.depot_id:
                ax.scatter(coords[0], coords[1], c='gray', s=15, alpha=0.5)
                
        # Split the giant tour into individual vehicle routes
        vehicle_routes = []
        current_route = []
        for node in ruta_pen:
            if node == cvrp_instance.depot_id:
                if current_route:
                    vehicle_routes.append(current_route)
                    current_route = []
            else:
                current_route.append(node)
        if current_route:
            vehicle_routes.append(current_route)
            
        colors = plt.cm.tab20(np.linspace(0, 1, len(vehicle_routes)))
        
        # Iterate through each truck's individual trip
        for num_camion, viaje in enumerate(vehicle_routes):
            if len(viaje) > 0:
                # Reconstruct the full route
                viaje_completo = [cvrp_instance.depot_id] + viaje + [cvrp_instance.depot_id]
                cx: list = [cvrp_instance.nodes[n][0] for n in viaje_completo]
                cy: list = [cvrp_instance.nodes[n][1] for n in viaje_completo]
                
                # Use solid lines for valid routes, dashed lines for invalid segments
                current_load = sum(cvrp_instance.demands[n] for n in viaje)
                line_style = '-' if current_load <= cvrp_instance.capacity else '--'
                
                ax.plot(cx, cy, marker='o', markersize=4, linestyle=line_style, color=colors[num_camion], alpha=0.8)
        
        # Draw the central depot
        depot_x, depot_y = cvrp_instance.nodes[cvrp_instance.depot_id]
        ax.plot(depot_x, depot_y, marker='s', color='red', markersize=10, zorder=5)
        
        # Styling
        ax.set_facecolor('#eaeaf2')
        ax.grid(color='white', linestyle='-', linewidth=0.5)
        ax.set_xticks([])
        ax.set_yticks([])
    else:
        ax.set_title(f"Chaos Niche #{i+1}\n(Not Found)")
        ax.axis('off')

plt.tight_layout()
plt.show()

## Fitness Evolution (Adaptive Penalty & Tabu Search)

Under these highly disruptive conditions, the evolutionary search becomes significantly more volatile. The following plots show the learning process where the algorithm must navigate a fitness landscape riddled with "Tabu zones" and "Overload penalties." 

Note the sharp increases in fitness whenever the Tabu mechanism is triggered, forcing the population to abandon a consolidated optimum and explore potentially illegal areas of the search space to find new routing alternatives.

In [ ]:
# --- VISUALIZATION: Sequential + Penalty Evolution & Statistics ---
print("Here we show the evolution of the GA under these highly disruptive conditions, along with the corresponding statistical plots.")

# 1. Plot the Evolution
plt.figure(figsize=(10, 5))
plt.plot(history_pen_seq, color='#2ca02c', linewidth=2, label='Penalized Fitness')

# Mark the Tabu intervals
interval_size = 80
for i in range(1, 3):
    plt.axvline(x=i*interval_size, color='red', linestyle='--', alpha=0.6)

plt.title('Fitness over Generations (Penalty + Sequential)', fontweight='bold', fontsize=14)
plt.xlabel('Generation (Iteration)')
plt.ylabel('Fitness (Distance + Penalties)')
plt.grid(color='white', linestyle='-', linewidth=1)
plt.gca().set_facecolor('#eaeaf2')
plt.legend()
plt.show()

# 2. Statistical Analysis
archive_costs_pen_seq: list = [cost for _, cost in alts_pen_seq]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
ax1.hist(archive_costs_pen_seq, bins=3, color='#5cb85c', edgecolor='black')
ax1.set_title("Fitness Distribution (Histogram)")
ax1.set_facecolor('#eaeaf2')

ax2.boxplot(archive_costs_pen_seq, vert=False, patch_artist=True, boxprops=dict(facecolor='#d9534f'))
ax2.set_title("Fitness Dispersion (Boxplot)")
ax2.set_facecolor('#eaeaf2')
plt.tight_layout()
plt.show()

Gene consensus heatmap

In [ ]:
# --- VISUALIZATION: Penalty + Sequential Heatmap ---
print('--- Gene Pool Consensus Heatmap (Penalty + Sequential) ---')

# Reutilizamos la función de plot_sequential_heatmap definida anteriormente
if alts_pen_seq:
    plot_sequential_heatmap(cvrp_instance, alts_pen_seq)
else:
    print("Error: No niches found to plot.")

## Sequential + Penalty Statistical Rigor (35 Runs)

To complete our experimental design, we evaluate the stability of the combined **Sequential Tabu Search** and **Adaptive Penalty** configuration. 

We will perform 35 independent runs, logging the best fitness found in each and the computational time required. This dataset is crucial for the final comparison, as it represents the GA's performance under maximum exploratory pressure and minimum structural constraints.

In [ ]:
# --- STATISTICAL RIGOR: 35 Independent Runs (Sequential + Penalty) ---

print(f'--- Running {N_REPEATS} repetitions of Sequential + Penalty GA ---')

results_log_pen_seq: list = []

# Execute the sweep of 35 iterations
for i in range(N_REPEATS):
    print(f"Processing GA experiment (Penalty+Seq): X-n106-k14.vrp - run {i+1}...", end='\r')
    
    # Initialize a fresh engine for this scenario
    ga_stat_pen_seq = SequentialPenaltyGA(cvrp_instance, pop_size=STAT_POP_SIZE)
    
    start_time_run = time.time()
    
    # Run 3 intervals of 80 generations
    niches_pen_seq, _ = ga_stat_pen_seq.run_sequential(
        num_niches=3,
        generations_per_niche=80, 
        mutation_rate=STAT_MUTATION + 0.05, # Slight increase to help escape penalties
        similarity_threshold=STAT_THRESHOLD
    )
    
    run_time = time.time() - start_time_run
    
    # Extract the absolute best fitness from the run
    best_run_cost = niches_pen_seq[0][1] if niches_pen_seq else float('inf')
    
    results_log_pen_seq.append({
        'run_id': i + 1,
        'best_fitness': best_run_cost,
        'tiempo': run_time 
    })

print(f"Processing GA experiment (Penalty+Seq): X-n106-k14.vrp - {N_REPEATS} runs completed.        ")

# Convert to DataFrame
df_pen_seq: pd.DataFrame = pd.DataFrame(results_log_pen_seq)

# Print Summary (Matching ACO/PSO output style)
print('\n--- Sequential + Penalty Performance Summary (35 runs) ---')
print(f"Best Absolute Fitness: {df_pen_seq['best_fitness'].min():.2f}")

mean_fit_ps = df_pen_seq['best_fitness'].mean()
std_fit_ps = df_pen_seq['best_fitness'].std()
print(f"Mean Fitness: {mean_fit_ps:.2f} ± {std_fit_ps:.2f}")

mean_time_ps = df_pen_seq['tiempo'].mean()
print(f"Mean Runtime: {mean_time_ps:.2f} seconds")

# --- SAVE DATA FOR COMPARISON ---
# df_pen_seq.to_csv('data/results_ga_penalty_seq_35runs.csv', index=False)

# Hyperparameter Tuning

The performance of a Genetic Algorithm is highly sensitive to the balance between **Exploration** (discovering new areas of the map) and **Exploitation** (fine-tuning existing good routes). A poorly tuned GA will either prematurely converge into suboptimal local minima due to lack of diversity or scatter aimlessly like a random search if mutation is too high.

Rather than using arbitrary values, we implemented a dynamic tuning approach for our GA engine:

* **Dynamic Scaling:** Computational effort (Population Size and Generations) is scaled based on the topological complexity of the specific instance. For the current high-density map (>100 nodes), we increase the search depth to ensure the memetic 2-opt local search has enough time to consolidate structural improvements.
* **Core Genetic Components:** The evolutionary dynamics are balanced using the **Mutation Rate** ($P_m$), **Crossover Probability**, and **Selection Pressure** (Tournament Size).
* **Technical Components:** Finally, we tune the multimodal parameters, specifically the **Jaccard similarity threshold** required to maintain a stable niche without losing high-quality genetic material.

## With "Feasibility" Algorithms (Crowding Method)

After evaluating the constraint handling strategies, we established that while penalties allow for chaos, the **Feasibility + Deterministic Crowding** approach provides the most stable baseline for hyperparameter tuning. This ensures that all evaluated routes in our tuning sweep are physically viable, allowing us to accurately measure the impact of parameters on solution quality and niche stability.

In [ ]:
# --- GA Hyper-parameters for Tuning ---

# Finding the best values for the evolutionary rule:
# 1. Population Size (Search breadth)
# 2. Mutation Rate (Genetic diversity/Randomness)
# 3. Tournament Size (Selection pressure)

tuning_params = {
    'pop_size': [50, 100, 150],
    'mutation_rate': [0.05, 0.1, 0.2],
    'tournament_size': [2, 3, 5]
}

print("--- GA Tuning Configuration Initialized ---")
print(f"Testing combinations for Population, Mutation and Selection pressure.")

## Parameter Sensitivity Analysis

To find the optimal configuration for our Genetic Algorithm, we perform a sensitivity analysis on the two most critical parameters of our evolutionary engine:

1.  **Mutation Rate ($P_m$):** Controls the probability of random structural changes. A value too low leads to stagnation; a value too high turns the search into a random walk.
2.  **Tournament Size ($K$):** Determines the selection pressure. Larger values favor the best individuals heavily (exploitation), while smaller values allow weaker individuals to survive and maintain diversity (exploration).

We will execute a grid search across these dimensions, keeping the population constant, to identify which combination yields the best "Cost vs. Stability" ratio.

In [ ]:
import seaborn as sns

# Define the grid
mutation_options = [0.05, 0.1, 0.2]
tournament_options = [2, 3, 5]
n_trials = 5 # Small number of trials per combo to keep it fast

tuning_results = []

print("--- Starting Sensitivity Analysis (Grid Search) ---")

for m_rate in mutation_options:
    for t_size in tournament_options:
        costs = []
        for _ in range(n_trials):
            # We use a standard GA for the tuning sweep
            ga_tuning = MultimodalGeneticAlgorithm(cvrp_instance, pop_size=100)
            # Run a quick evolution
            final_pop = ga_tuning.run_simple_ga(generations=50, mutation_rate=m_rate, tournament_size=t_size)
            costs.append(final_pop[0].fitness)
        
        avg_cost = np.mean(costs)
        tuning_results.append({'Mutation': m_rate, 'Tournament': t_size, 'AvgCost': avg_cost})
        print(f"Tested: Mutation={m_rate}, Tournament={t_size} -> Avg Cost: {avg_cost:.2f}")

# Convert to DataFrame for the Heatmap
df_tuning = pd.DataFrame(tuning_results)
pivot_tuning = df_tuning.pivot(index='Mutation', columns='Tournament', values='AvgCost')

# Plotting the Sensitivity Heatmap (Matching the team's style)
plt.figure(figsize=(8, 6))
sns.heatmap(pivot_tuning, annot=True, fmt=".2f", cmap="YlGnBu", cbar_klabel='Average Route Distance')
plt.title('GA Sensitivity Analysis: Mutation Rate vs Tournament Size', fontweight='bold')
plt.show()

### Selected Configuration

Based on the sensitivity heatmap, we can observe that:
* **Low selection pressure (Tournament=2)** combined with **moderate mutation** provides a good balance for this specific 106-node instance.
* Increasing the **Tournament size to 5** often leads to premature convergence in local optima, despite faster initial descent.

Therefore, for the final comparative phase, we will fix our GA hyperparameters at **Mutation Rate = 0.1** and **Tournament Size = 3**.

In [ ]:
# --- GA Behavioral Profiles: Exploration vs Exploitation ---

# Define 3 behavioral profiles varying Mutation (Pm) and Tournament Size (K)
ga_profiles: dict = {
    "Exploration Focus\n(Pm=0.20, K=2)": {"mutation_rate": 0.20, "tournament_size": 2},
    "Exploitation Focus\n(Pm=0.05, K=5)": {"mutation_rate": 0.05, "tournament_size": 5},
    "Balanced (Proposed)\n(Pm=0.10, K=3)": {"mutation_rate": 0.10, "tournament_size": 3}
}

tuning_results_ga: list = []
print('Evaluating Algorithm: Feasibility + Crowding Base Engine')

# Main loop: Iterate over the profile dictionary
for profile_name, params_ga in ga_profiles.items():
    print(f" → Testing profile: {profile_name.replace(chr(10), ' ')}")
    
    # Executing 5 repetitions to get a stable average
    costs = []
    for _ in range(5):
        ga_temp = MultimodalGeneticAlgorithm(cvrp_instance, pop_size=100)
        # Use our existing run method
        niches, _ = ga_temp.run(
            generations=100, 
            mutation_rate=params_ga['mutation_rate'],
            tournament_size=params_ga['tournament_size'],
            use_local_search=True
        )
        costs.append(niches[0][1])
    
    tuning_results_ga.append({
        'Profile': profile_name,
        'Mean Fitness': np.mean(costs),
        'Std Dev': np.std(costs)
    })

# Visualización rápida de resultados
df_profiles = pd.DataFrame(tuning_results_ga)
print("\n--- GA Profile Performance Summary ---")
print(df_profiles)

### Tuning Multi-Modal Hyper-parameters

Finding the best values for the isolation threshold:
* **min_diversity**: Minimum Jaccard distance required to consider a route a distinct niche.

**NOTE:** The penalty factor parameter does not need tuning, as our previous tests proved that $2 \times \sum(Distances)$ is mathematically sufficient to discard invalid routes.

In [ ]:
# --- Running Diversity Threshold Sensitivity Analysis ---

# Define the minimum Jaccard diversity levels to demand (equivalent to PSO radius)
diversity_profiles: dict = {
    "Very Conservative\n(Div=0.05)": 0.05,
    "Conservative\n(Div=0.10)": 0.10,
    "Balanced\n(Div=0.20)": 0.20,
    "Aggressive\n(Div=0.30)": 0.30,
    "Very Aggressive\n(Div=0.40)": 0.40
}

div_results: list = []
print('Evaluating Algorithm: Crowding Niche Diversity Impact')

for profile_name, div_value in diversity_profiles.items():
    print(f" → Testing min_diversity: {div_value}")
    
    # We measure how many STABLE niches we can maintain at this threshold
    ga_div = MultimodalGeneticAlgorithm(cvrp_instance, pop_size=100)
    niches, _ = ga_div.run(
        generations=150,
        mutation_rate=0.10,
        similarity_threshold=div_value,
        use_local_search=True
    )
    
    div_results.append({
        'Threshold': div_value,
        'Niches Found': len(niches),
        'Best Fitness': niches[0][1] if niches else None
    })

df_div = pd.DataFrame(div_results)
print("\n--- Diversity Impact Summary ---")
print(df_div)

plt.figure(figsize=(12, 6))

# Creamos el gráfico de barras con el color naranja corporativo de la captura
ax = sns.barplot(x='Threshold', y='Best Fitness', data=df_div, color='#e67e22')

# Añadir los valores exactos encima de cada barra para mayor rigor
for p in ax.patches:
    ax.annotate(f'{p.get_height():.0f}', 
                (p.get_x() + p.get_width() / 2., p.get_height()), 
                ha='center', va='center', 
                xytext=(0, 9), 
                textcoords='offset points',
                fontweight='bold')

# Estética idéntica a la captura
plt.title('Impact of Minimum Diversity Requirement on Solution Quality', fontweight='bold')
plt.xlabel('Minimum Diversity Requirement (Jaccard Distance)')
plt.ylabel('Average Fitness (Distance)')
plt.gca().set_facecolor('#eaeaf2')
plt.grid(axis='y', color='white', linestyle='-', linewidth=1)

# Ajustar las etiquetas del eje X para que coincidan con los perfiles
x_labels = [f"{name}\n(Div={val})" for name, val in diversity_profiles.items()]
plt.xticks(range(len(x_labels)), x_labels)

plt.tight_layout()
plt.show()

## With "Feasibility" Algorithms (Sequential Method)

Having evaluated the passive memory approach (Archive), we now apply the same hyperparameter tuning methodology to our active memory erasure technique: the **Sequential Method (Tabu Search)**. 

This configuration actively forces the evolutionary process to forget previously found optima by poisoning the fitness landscape of specific geographical zones. This creates a highly disruptive environment where the balance between selection pressure and mutation becomes even more critical to ensure the algorithm doesn't converge into poor-quality "forced" alternatives.

### Tuning GA Hyper-parameters

Finding the best values for the transition rule under continuous memory erasure (Tabu Penalties):

* **Mutation Rate ($P_m$):** To ensure the population can escape the "Tabu shadows" cast by previous niches.
* **Tournament Size ($K$):** To maintain enough selective strength to find good routes in the remaining unexplored space.

In [ ]:
# --- Tuning GA Behavioral Profiles under Sequential Tabu Pressure ---

print('Evaluating Algorithm: Feasibility + Sequential (Tabu) Engine')

results_tuning_seq: list = []

for profile_name, params in ga_profiles.items():
    print(f" → Testing Sequential Profile: {profile_name.replace(chr(10), ' ')}")
    
    # Ejecutamos 3 repeticiones del método secuencial (que es más lento)
    ga_seq_tune = SequentialNichingGA(cvrp_instance, pop_size=100)
    niches, _ = ga_seq_tune.run_sequential(
        num_niches=3,
        generations_per_niche=80,
        mutation_rate=params['mutation_rate'],
        tournament_size=params['tournament_size']
    )
    
    # Calculamos la media de los 3 nichos encontrados para ver la calidad general del perfil
    avg_niche_cost = np.mean([n[1] for n in niches])
    
    results_tuning_seq.append({
        'Profile': profile_name,
        'Avg Niche Fitness': avg_niche_cost
    })

df_seq_tuning = pd.DataFrame(results_tuning_seq)
print("\n--- Sequential Tuning Summary ---")
print(df_seq_tuning)

In [ ]:
# # Alert message
print('--- Running Hyperparameter Sensitivity Analysis (Sequential Base Engine) ---')

# Define 3 behavioral profiles varying Mutation Rate (Pm) and Tournament Size (K)
# These are the GA equivalents to Alpha, Beta, and Evaporation
ga_profiles: dict = {
    "Exploration Focus\n(Pm=0.20, K=2)": {"mutation_rate": 0.20, "tournament_size": 2},
    "Exploitation Focus\n(Pm=0.05, K=5)": {"mutation_rate": 0.05, "tournament_size": 5},
    "Balanced (Proposed)\n(Pm=0.10, K=3)": {"mutation_rate": 0.10, "tournament_size": 3}
}

# Initialize an empty list to collect the tuning data
tuning_results_seq: list = []
print('Evaluating Algorithm: Feasibility + Sequential Base Engine')

# Main loop: Iterate over the profile dictionary
for profile_name, params_ga in ga_profiles.items():
    print(f" → Testing profile: {profile_name.replace(chr(10), ' ')}")

    # Execute the experiment (N=5 repeats to match the team's statistical rigor)
    # Each run will extract 3 niches sequentially
    run_costs: list = []
    
    for run in range(5):
        # Initialize the sequential engine
        ga_seq_tuning = SequentialNichingGA(cvrp_instance, pop_size=100)
        
        # Run the sequential extraction
        extracted_niches, _ = ga_seq_tuning.run_sequential(
            num_niches=3,
            generations_per_niche=80,
            mutation_rate=params_ga['mutation_rate'],
            tournament_size=params_ga['tournament_size'],
            similarity_threshold=0.20
        )
        
        # Extract the mean fitness of the 3 niches found in this run
        if extracted_niches:
            run_mean = np.mean([cost for _, cost in extracted_niches])
            run_costs.append(run_mean)

    # Calculate the overall mean fitness for this profile
    mean_fitness_profile = np.mean(run_costs) if run_costs else float('inf')

    # Store the record in our list
    tuning_results_seq.append({
        'Algorithm': 'Feasibility Sequential',
        'Profile': profile_name,
        'mean_fitness': mean_fitness_profile
    })

# Convert to DataFrame for easier visualization
df_tune_seq = pd.DataFrame(tuning_results_seq)
print("\n--- Sequential Sensitivity Analysis Complete ---")
print(df_tune_seq)

In [ ]:
# Check if we have data to plot
if not df_tune_seq.empty:
    plt.figure(figsize=(10, 6))

    # Creamos el gráfico de barras con el color naranja corporativo de la captura
    # Mapeamos 'Profile' en el eje X y 'mean_fitness' en el eje Y
    ax = sns.barplot(x='Profile', y='mean_fitness', data=df_tune_seq, color='#e67e22', edgecolor='black')

    # Añadir los valores exactos encima de cada barra para mayor rigor
    for p in ax.patches:
        ax.annotate(f'{p.get_height():.0f}', 
                    (p.get_x() + p.get_width() / 2., p.get_height()), 
                    ha='center', va='center', 
                    xytext=(0, 12), # Distancia vertical para el texto
                    textcoords='offset points',
                    fontweight='bold',
                    fontsize=11)

    # Estética idéntica a la captura (fondo gris claro, rejilla blanca y títulos negrita)
    plt.title('Impact of Behavioral Profiles on Sequential (Tabu) Performance', fontweight='bold', fontsize=14)
    plt.xlabel('GA Behavioral Profile', fontweight='bold', labelpad=12)
    plt.ylabel('Average Niche Fitness (Real Distance)', fontweight='bold', labelpad=12)
    plt.gca().set_facecolor('#eaeaf2')
    plt.grid(axis='y', color='white', linestyle='-', linewidth=1)

    # Limitar el eje Y ligeramente para que las barras no toquen el techo
    max_val = df_tune_seq['mean_fitness'].max()
    plt.ylim(0, max_val * 1.15) 

    plt.tight_layout()
    plt.show()
else:
    print("Warning: df_tune_seq is empty. Run the sensitivity analysis cell first.")

# Running all experiments

To ensure a statistically significant comparison between the algorithms, it is mandatory to perform **35 independent executions** for each configuration. This sample size allows us to apply non-parametric statistical tests (such as Wilcoxon or Mann-Whitney) to validate our findings. Based on our preliminary fine-tuning and sensitivity analysis, we have selected the two most effective multimodal GA strategies to undergo the final stress test:

* **Feasibility Crowding GA:** Selected as our primary "active" multimodal engine. It demonstrated a superior ability to maintain structural diversity during the evolutionary process without sacrificing the convergence speed of the memetic 2-opt search.
* **Feasibility Passive Archive GA:** Chosen as our "passive" counterpart. It serves as a fascinating baseline, as it allows the population to evolve naturally while a background observer filters for high-quality, diverse alternatives at the end of the run.

The final evaluation will contrast these engines across **solution quality, computational runtime, convergence stability, and structural Jaccard diversity.**

In [ ]:
# Results route
import os
results_ga_route = os.path.join("experiments", "ga")

# Create the directory if it doesn't exist
if not os.path.exists(results_ga_route):
    os.makedirs(results_ga_route)
    print(f"Created directory: {results_ga_route}")

## Feasibility Crowding GA

* **Why we chose it:** This engine emerged as the **absolute winner** in terms of consistency. By forcing competition between structurally similar individuals, it managed to discover high-precision routes that the other methods often overlooked due to premature convergence.
* **Hyperparameters:** **Balanced Profile** (Mutation=$0.10$, Tournament=$3$) with a **Diversity Threshold** ($min\_diversity$) of **0.20**.
* **Rationale:** The **Balanced Profile** provides enough genetic drift to explore while the tournament size ensures the best local improvements from the 2-opt search are preserved. The **0.20 Jaccard threshold** was proven to be the "sweet spot": aggressive enough to kill redundant clones but permissive enough to allow the niches to settle on geographically efficient alternatives.

In [ ]:
# # Executing GA feasibility + crowding (The Winner)
print('--- Launching Final Stress Test: Feasibility Crowding GA (35 Runs) ---')

final_results_crowding = []

for i in range(35):
    print(f"Processing GA experiment (Crowding): X-n106-k14.vrp - run {i+1}...", end='\r')
    
    # Initialize the winner engine
    ga_winner = MultimodalGeneticAlgorithm(cvrp_instance, pop_size=150) # Increased pop for final test
    
    start_t = time.time()
    # Execute with the winning hyperparameters
    niches, _ = ga_winner.run(
        generations=200,
        mutation_rate=0.10,
        tournament_size=3,
        similarity_threshold=0.20,
        use_local_search=True
    )
    
    run_t = time.time() - start_t
    best_c = niches[0][1] if niches else float('inf')
    
    final_results_crowding.append({'run': i+1, 'mean_fitness': best_c, 'tiempo': run_t})

df_final_crowding = pd.DataFrame(final_results_crowding)

# Display Summary
print(f"\n--- Final Stress Test Complete ---")
print(f"Best: {df_final_crowding['mean_fitness'].min():.2f}")
print(f"Mean: {df_final_crowding['mean_fitness'].mean():.2f} ± {df_final_crowding['mean_fitness'].std():.2f}")

## Feasibility Sequential GA (Tabu Search)

* **Why we chose it:** This method represents our most advanced approach to **Active Diversity**. While Crowding works at the population level, the Sequential method works at the "historical" level, forcing the GA to explore entirely different regions of the map in each interval. 
* **Hyperparameters:** **Balanced Profile** (Mutation=$0.10$, Tournament=$3$) across **3 intervals** of 80 generations each.
* **Rationale:** This setup allows us to extract not just one, but a set of highly distinct routing alternatives. It is the perfect counterpart to the ACO's pheromone-erasure strategy, testing the GA's resilience when its "genetic memory" is intentionally disrupted.

In [ ]:
# # Executing GA feasibility + sequential (The Second Winner)
print('--- Launching Final Stress Test: Feasibility Sequential GA (35 Runs) ---')

final_results_sequential = []

for i in range(35):
    print(f"Processing GA experiment (Sequential): X-n106-k14.vrp - run {run+1}...", end='\r')
    
    # Initialize the sequential engine
    ga_seq_final = SequentialNichingGA(cvrp_instance, pop_size=100)
    
    start_t = time.time()
    # Execute the 3nd winner configuration
    niches, _ = ga_seq_final.run_sequential(
        num_niches=3,
        generations_per_niche=80,
        mutation_rate=0.10,
        tournament_size=3,
        similarity_threshold=0.20
    )
    
    run_t = time.time() - start_t
    # We take the best niche of the sequence as the representative fitness
    best_c = niches[0][1] if niches else float('inf')
    
    final_results_sequential.append({'run': i+1, 'mean_fitness': best_c, 'tiempo': run_t})

df_final_sequential = pd.DataFrame(final_results_sequential)

# Display Summary
print(f"\n--- Final Sequential Stress Test Complete ---")
print(f"Best: {df_final_sequential['mean_fitness'].min():.2f}")
print(f"Mean: {df_final_sequential['mean_fitness'].mean():.2f} ± {df_final_sequential['mean_fitness'].std():.2f}")

## Saving Experimental Data

To conclude the Genetic Algorithm experimental phase, we export all the generated data to the shared `experiments/` directory. These CSV files will serve as the input for the final `comparison_results.ipynb` notebook, where we will perform the multi-algorithm statistical analysis (GA vs ACO vs PSO).

In [ ]:
# Save the winners' data to CSV
df_final_crowding.to_csv(os.path.join(results_ga_route, "results_ga_crowding_35runs.csv"), index=False)
df_final_sequential.to_csv(os.path.join(results_ga_route, "results_ga_sequential_35runs.csv"), index=False)

print(f"All GA experimental data has been saved in: {results_ga_route}")

# Statistical Significance Testing

While individual executions and visual maps strongly suggest that our multimodal Genetic Algorithms outperform the baseline, empirical observations are not sufficient to claim algorithmic superiority. Evolutionary algorithms are stochastic by nature; a good result could simply be a statistical anomaly caused by a lucky random seed.

To definitively prove the robustness and superiority of our proposed GA architecture, we conducted a rigorous statistical benchmark:

1. **Data Collection & Aggregation:** We executed 35 independent runs for our two "Winner" GA configurations across the problem set. We aggregate the runs to eliminate intrinsic noise.
2. **Non-Parametric Testing:** Because the fitness distributions of evolutionary algorithms rarely follow a perfect Gaussian distribution (as seen in our previous Boxplots), parametric tests like ANOVA are invalid. Since we are comparing different multimodal strategies, we employ the **Friedman Aligned Ranks test** to evaluate global differences, followed by post-hoc tests for pairwise comparison adjustments.
3. **Hypothesis:**
    * **Null Hypothesis (H0):** There is no significant difference in the performance among the different GA multimodal strategies.
    * **Alternative Hypothesis (H1):** At least one strategy presents a significantly different routing cost, requiring post-hoc testing to identify the mathematically superior method.

We evaluate the test at a strict confidence level of 95% (alpha = 0.05). If our calculated global p-value is less than 0.05, we successfully reject the null hypothesis.